# Kinship Verification — KinFaceW-II + AdaFace IR-101

| | |
|---|---|
| **Dataset** | KinFaceW-II (fd / fs / md / ms, ~2 000 pre-labelled pairs) |
| **Backbone** | AdaFace IR-101 webface12m — **frozen** feature extractor |
| **Head** | 4-component Siamese MLP (diff, product, cosine, L2-dist → 1026-D input) |
| **Loss** | Label-Smoothed BCE (s=0.05) + Youden's J threshold calibration |
| **Platform** | Google Colab T4 GPU — fully automated end-to-end |

> **Note:** If the dataset auto-download fails (site may require registration),
> follow the printed message to manually place `KinFaceW-II.zip` at `/content/`.


In [1]:
# ──────────────────────────────────────────────────────────────────────────
# Install Dependencies
# ──────────────────────────────────────────────────────────────────────────

import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("huggingface_hub>=0.20.0", "safetensors")
pip("scipy", "scikit-learn", "tqdm", "Pillow")
pip("fvcore")
print("✅  Dependencies ready.")

✅  Dependencies ready.


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# Global Configuration
# ──────────────────────────────────────────────────────────────────────────

import os
from dataclasses import dataclass, field
@dataclass
class Config:
    # --- Paths (unchanged) ---
    DATA_DIR:       str  = "/content/kinfacew2"
    CACHE_PATH:     str  = "/content/cache_embeddings.npy"
    CHECKPOINT_DIR: str  = "/content/checkpoints"
    BEST_MODEL:     str  = "/content/checkpoints/best_model.pt"
    MODEL_DIR:      str  = "/content/adaface_ir101"

    # --- Backbone (unchanged) ---
    HF_MODEL_ID: str = "minchul/cvlface_adaface_ir101_webface12m"
    EMBED_DIM:   int = 512
    IMG_SIZE:    int = 112

    # --- Dataset (unchanged) ---
    RELATIONS:   tuple = ("father-dau", "father-son", "mother-dau", "mother-son")
    TRAIN_RATIO: float = 0.80

    # --- Head: SMALLER + MORE REGULARIZATION ---
    HEAD_HIDDEN:  list  = field(default_factory=lambda: [64])  # was [128, 64]
    HEAD_DROPOUT: float = 0.60                                  # was 0.45

    # --- Training ---
    BATCH_SIZE:   int   = 32          # was 64 — smaller = more gradient noise (helps generalize)
    NUM_EPOCHS:   int   = 200         # was 120 — early stopping handles actual cutoff
    LR:           float = 1e-3        # was 3e-4 — higher LR + more decay = less overfit
    WEIGHT_DECAY: float = 1e-2        # was 5e-3 — stronger L2 regularization

    # --- Scheduler / Early Stopping ---
    LR_PATIENCE:  int   = 10          # was 8
    LR_FACTOR:    float = 0.5
    ES_PATIENCE:  int   = 35          # was 25 — give it more time to find a better plateau

    # --- Misc (unchanged) ---
    SEED:          int = 42
    NUM_WORKERS:   int = 2
    EXTRACT_BATCH: int = 128

CFG = Config()
for d in [CFG.DATA_DIR, os.path.dirname(CFG.CACHE_PATH),
          CFG.CHECKPOINT_DIR, CFG.MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅  Config initialized.")
print(f"    Data     : {CFG.DATA_DIR}")
print(f"    Backbone : {CFG.HF_MODEL_ID}")

✅  Config initialized.
    Data     : /content/kinfacew2
    Backbone : minchul/cvlface_adaface_ir101_webface12m


In [3]:
# ──────────────────────────────────────────────────────────────────────────
# Reproducibility + Device
# ──────────────────────────────────────────────────────────────────────────

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(CFG.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅  Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"    GPU    : {torch.cuda.get_device_name(0)}")
    print(f"    VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅  Device : cuda
    GPU    : Tesla T4
    VRAM   : 15.6 GB


In [4]:
# ──────────────────────────────────────────────────────────────────────────
# Download & Extract KinFaceW-II
# ──────────────────────────────────────────────────────────────────────────

import zipfile
import glob
import urllib.request

DATASET_URL = "https://www.kinfacew.com/dataset/KinFaceW-II.zip"
ZIP_PATH    = "/content/KinFaceW-II.zip"


def _report(block, block_size, total):
    downloaded = min(block * block_size, total)
    print(f"\r    {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="", flush=True)


def download_kinfacew2(url: str, zip_path: str, data_dir: str) -> None:
    if glob.glob(os.path.join(data_dir, "**", "pairs.mat"), recursive=True):
        print("✅  Dataset already present — skipping download.")
        return

    if not os.path.exists(zip_path):
        print("⬇️   Downloading KinFaceW-II (~50 MB)…")
        try:
            urllib.request.urlretrieve(url, zip_path, reporthook=_report)
            print()
        except Exception as exc:
            raise RuntimeError(
                f"Auto-download failed: {exc}\n\n"
                "➜  Please download manually from:\n"
                "   https://www.kinfacew.com/download.html\n"
                "   and place KinFaceW-II.zip at /content/KinFaceW-II.zip"
            ) from exc

    print("📦  Extracting…")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)

    n_imgs = len(glob.glob(os.path.join(data_dir, "**", "*.jpg"), recursive=True))
    print(f"✅  Extracted — {n_imgs:,} images found in {data_dir}")


download_kinfacew2(DATASET_URL, ZIP_PATH, CFG.DATA_DIR)

mat_files = sorted(glob.glob(os.path.join(CFG.DATA_DIR, "**", "pairs.mat"), recursive=True))
print(f"\n📂  Found {len(mat_files)} pairs.mat file(s):")
for f in mat_files:
    print(f"    {f}")

⬇️   Downloading KinFaceW-II (~50 MB)…
    4.4 / 4.4 MB
📦  Extracting…
✅  Extracted — 2,000 images found in /content/kinfacew2

📂  Found 0 pairs.mat file(s):


In [5]:
# ──────────────────────────────────────────────────────────────────────────
# Load AdaFace IR-101 Backbone (CVLFace / HuggingFace)  ── v3 FINAL
# ──────────────────────────────────────────────────────────────────────────

import os, sys, shutil, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download


# ── Download (unchanged — works perfectly) ────────────────────────────────
def _cvlface_download(repo_id: str, local_dir: str, hf_token=None) -> None:
    os.makedirs(local_dir, exist_ok=True)
    files_txt = os.path.join(local_dir, "files.txt")
    if not os.path.exists(files_txt):
        hf_hub_download(repo_id, "files.txt", token=hf_token,
                        local_dir=local_dir, local_dir_use_symlinks=False)
    with open(files_txt) as f:
        extra_files = [l.strip() for l in f if l.strip()]
    for fname in extra_files + ["config.json", "wrapper.py", "model.safetensors"]:
        dest = os.path.join(local_dir, fname)
        if not os.path.exists(dest):
            print(f"    ⬇️  {fname}")
            hf_hub_download(repo_id, fname, token=hf_token,
                            local_dir=local_dir, local_dir_use_symlinks=False)


# ── Load: bypass AutoModel, use repo's own loader directly ───────────────
def _cvlface_load_direct(local_dir: str) -> nn.Module:
    """
    AutoModel.from_pretrained fails because CVLFaceRecognitionModel doesn't
    implement the full PreTrainedModel interface in newer transformers versions.

    The fix: replicate exactly what wrapper.py does internally:
      1. Load model config via OmegaConf
      2. Build the iResNet backbone via get_model()
      3. Load pretrained weights via load_state_dict_from_path()

    We already know steps 2-3 work (917/917 keys matched in your output).
    We just stop before transformers tries to finalize.
    """
    cwd = os.getcwd()
    try:
        os.chdir(local_dir)                      # needed for relative weight path
        sys.path.insert(0, local_dir)

        from omegaconf import OmegaConf
        from models import get_model             # repo's own models package

        model_conf = OmegaConf.load("pretrained_model/model.yaml")
        backbone = get_model(model_conf)         # prints "Loaded iResNet model"
        backbone.load_state_dict_from_path(      # prints "917/917 matched"
            "pretrained_model/model.pt"
        )
    finally:
        os.chdir(cwd)
        if local_dir in sys.path:
            sys.path.remove(local_dir)

    return backbone


# ── Public entry point ────────────────────────────────────────────────────
def load_adaface_backbone(
    hf_model_id: str,
    model_dir: str,
    hf_token=None,
    force_download: bool = False,
) -> nn.Module:
    if force_download and os.path.exists(model_dir):
        shutil.rmtree(model_dir)
        print(f"🗑️   Cleared stale cache: {model_dir}")

    print(f"⬇️   Fetching {hf_model_id} → {model_dir}")
    _cvlface_download(hf_model_id, model_dir, hf_token)
    print("    Download complete.")

    backbone = _cvlface_load_direct(model_dir)

    backbone.eval()
    for p in backbone.parameters():
        p.requires_grad_(False)

    print("✅  Backbone loaded (CVLFace direct, no AutoModel).")
    return backbone


# ── Load ──────────────────────────────────────────────────────────────────
BACKBONE = load_adaface_backbone(
    CFG.HF_MODEL_ID,
    CFG.MODEL_DIR,
    hf_token=None,
    force_download=False,   # files already cached — no re-download needed
).to(DEVICE)


# ── Smoke test ────────────────────────────────────────────────────────────
with torch.no_grad():
    _x   = torch.randn(4, 3, CFG.IMG_SIZE, CFG.IMG_SIZE, device=DEVICE)
    _out = BACKBONE(_x)
    _emb = _out[0] if isinstance(_out, (list, tuple)) else _out
    if isinstance(_emb, dict):
        _emb = next(iter(_emb.values()))
    _emb = F.normalize(_emb, p=2, dim=1)
    print(f"✅  Backbone smoke test — shape: {_emb.shape},  norm≈{_emb.norm(dim=1).mean():.4f}")

⬇️   Fetching minchul/cvlface_adaface_ir101_webface12m → /content/adaface_ir101


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


files.txt:   0%|          | 0.00/350 [00:00<?, ?B/s]

    ⬇️  README.md


README.md: 0.00B [00:00, ?B/s]

...

    ⬇️  wrapper.py


wrapper.py:   0%|          | 0.00/766 [00:00<?, ?B/s]

    ⬇️  model.safetensors


model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

    Download complete.
Loaded iResNet model
compatible keys in state_dict 917 / 917
Check


<All keys matched successfully>
Loaded pretrained model from pretrained_model/model.pt
✅  Backbone loaded (CVLFace direct, no AutoModel).
✅  Backbone smoke test — shape: torch.Size([4, 512]),  norm≈1.0000


In [6]:
# ──────────────────────────────────────────────────────────────────────────
# Extract & Cache All Face Embeddings (Batched GPU)
# ──────────────────────────────────────────────────────────────────────────

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


FACE_TRANSFORM = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    # AdaFace / ArcFace standard normalization (maps [0,255]→[-1,1])
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


class _ImageDataset(Dataset):
    """Lightweight dataset for batch embedding extraction."""

    def __init__(self, paths: list, transform):
        self.paths     = paths
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert("RGB")
            return self.transform(img), path
        except Exception:
            return torch.zeros(3, CFG.IMG_SIZE, CFG.IMG_SIZE), path


def _extract(backbone: nn.Module, img_paths: list, device: torch.device) -> dict:
    ds     = _ImageDataset(img_paths, FACE_TRANSFORM)
    loader = DataLoader(ds, batch_size=CFG.EXTRACT_BATCH, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)
    cache  = {}
    with torch.no_grad():
        for imgs, paths in tqdm(loader, desc="Extracting embeddings", unit="batch"):
            imgs = imgs.to(device)
            out  = backbone(imgs)
            embs = out[0] if isinstance(out, (list, tuple)) else out
            if isinstance(embs, dict):
                embs = next(iter(embs.values()))
            embs = F.normalize(embs, p=2, dim=1).cpu().numpy().astype(np.float32)
            for path, emb in zip(paths, embs):
                cache[path] = emb
    return cache


def build_embed_cache(backbone: nn.Module, cfg: Config) -> dict:
    all_paths = sorted(glob.glob(
        os.path.join(cfg.DATA_DIR, "**", "*.jpg"), recursive=True
    ) + glob.glob(
        os.path.join(cfg.DATA_DIR, "**", "*.png"), recursive=True
    ))
    print(f"Found {len(all_paths):,} images.")

    if os.path.exists(cfg.CACHE_PATH):
        cached = np.load(cfg.CACHE_PATH, allow_pickle=True).item()
        missing = [p for p in all_paths if p not in cached]
        if not missing:
            print(f"✅  Loaded cache — {len(cached):,} embeddings.")
            return cached
        print(f"    Cache incomplete ({len(missing)} missing). Rebuilding…")

    print(f"⏳  Extracting embeddings via GPU batches (batch={cfg.EXTRACT_BATCH})…")
    cache = _extract(backbone, all_paths, DEVICE)
    np.save(cfg.CACHE_PATH, cache)
    print(f"✅  Cache saved — {len(cache):,} embeddings → {cfg.CACHE_PATH}")
    return cache


EMBED_CACHE = build_embed_cache(BACKBONE, CFG)

Found 2,000 images.
⏳  Extracting embeddings via GPU batches (batch=128)…


Extracting embeddings:   0%|          | 0/16 [00:00<?, ?batch/s]

✅  Cache saved — 2,000 embeddings → /content/cache_embeddings.npy


In [7]:
import shutil

BASE = "/content/kinfacew2/KinFaceW-II"
IMGS = f"{BASE}/images"
META = f"{BASE}/meta_data"

# actual folder name → relation code
FOLDER_MAP = {
    "fd": "father-dau",
    "fs": "father-son",
    "md": "mother-dau",
    "ms": "mother-son",
}

# Copy each fd_pairs.mat → images/father-dau/pairs.mat
for code, folder in FOLDER_MAP.items():
    src = os.path.join(META, f"{code}_pairs.mat")
    dst = os.path.join(IMGS, folder, "pairs.mat")
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f"✅  {code}_pairs.mat  →  {folder}/pairs.mat")
    else:
        print(f"✔️   {folder}/pairs.mat already exists")

# Point DATA_DIR at images/ and use full folder names as relations
CFG.DATA_DIR  = IMGS
CFG.RELATIONS = tuple(FOLDER_MAP.values())   # ('father-dau', 'father-son', 'mother-dau', 'mother-son')

print(f"\nCFG.DATA_DIR  → {CFG.DATA_DIR}")
print(f"CFG.RELATIONS → {CFG.RELATIONS}")

✅  fd_pairs.mat  →  father-dau/pairs.mat
✅  fs_pairs.mat  →  father-son/pairs.mat
✅  md_pairs.mat  →  mother-dau/pairs.mat
✅  ms_pairs.mat  →  mother-son/pairs.mat

CFG.DATA_DIR  → /content/kinfacew2/KinFaceW-II/images
CFG.RELATIONS → ('father-dau', 'father-son', 'mother-dau', 'mother-son')


In [8]:
# ──────────────────────────────────────────────────────────────────────────
# Load KinFaceW-II Pairs
# ──────────────────────────────────────────────────────────────────────────

import scipy.io as sio


def _find_data_root(base: str) -> str:
    """
    KinFaceW-II.zip may extract to KinFaceW-II/data/ or data/ depending on platform.
    Walk up to find the directory that contains fd/, fs/, md/, ms/.
    """
    for suffix in ["KinFaceW-II/data", "KinFaceW_II/data", "data", ""]:
        candidate = os.path.join(base, suffix) if suffix else base
        if os.path.isdir(candidate) and any(
            os.path.isdir(os.path.join(candidate, r))
            for r in ("fd", "fs", "md", "ms")
        ):
            return candidate
    return base


def _load_relation(rel_dir: str) -> list:
    import scipy.io as sio

    mat_path = os.path.join(rel_dir, "pairs.mat")
    if not os.path.exists(mat_path):
        hits = glob.glob(os.path.join(rel_dir, "**", "pairs.mat"), recursive=True)
        if not hits:
            print(f"    ⚠️  pairs.mat not found in {rel_dir}")
            return []
        mat_path = hits[0]

    raw = sio.loadmat(mat_path)["pairs"]   # shape (N, 4)

    pairs = []
    for row in raw:
        # col 0 = label, col 1 = fold, col 2 = img_a, col 3 = img_b
        label  = int(row[0].flat[0])                  # 1 = kin, 0 or other = non-kin
        name_a = str(row[2].flat[0]).strip()
        name_b = str(row[3].flat[0]).strip()

        label  = 1 if label == 1 else 0
        img_a  = os.path.join(rel_dir, name_a)
        img_b  = os.path.join(rel_dir, name_b)

        if img_a in EMBED_CACHE and img_b in EMBED_CACHE:
            pairs.append((img_a, img_b, label))

    return pairs

def load_all_pairs(data_dir: str, relations: tuple) -> list:
    data_root = _find_data_root(data_dir)
    print(f"📂  Data root : {data_root}")

    all_pairs = []
    for rel in relations:
        rel_dir   = os.path.join(data_root, rel)
        if not os.path.isdir(rel_dir):
            print(f"    ⚠️  Directory not found: {rel_dir}")
            continue
        rel_pairs = _load_relation(rel_dir)
        pos = sum(l for _, _, l in rel_pairs)
        print(f"    {rel}: {len(rel_pairs):>4} pairs  "
              f"({pos} kin / {len(rel_pairs) - pos} non-kin)")
        all_pairs.extend(rel_pairs)

    pos_total = sum(l for _, _, l in all_pairs)
    print(f"\n✅  Total: {len(all_pairs)} pairs  "
          f"({pos_total} kin / {len(all_pairs) - pos_total} non-kin)")
    return all_pairs


ALL_PAIRS = load_all_pairs(CFG.DATA_DIR, CFG.RELATIONS)
assert len(ALL_PAIRS) > 0, "❌  No pairs loaded — check dataset path and structure."

📂  Data root : /content/kinfacew2/KinFaceW-II/images
    father-dau:  500 pairs  (100 kin / 400 non-kin)
    father-son:  500 pairs  (100 kin / 400 non-kin)
    mother-dau:  500 pairs  (100 kin / 400 non-kin)
    mother-son:  500 pairs  (100 kin / 400 non-kin)

✅  Total: 2000 pairs  (400 kin / 1600 non-kin)


In [9]:
# ──────────────────────────────────────────────────────────────────────────
# Dataset, DataLoader, Train/Val Split
# ──────────────────────────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

labels = [l for _, _, l in ALL_PAIRS]
train_pairs, val_pairs = train_test_split(
    ALL_PAIRS,
    test_size    = 1.0 - CFG.TRAIN_RATIO,
    stratify     = labels,
    random_state = CFG.SEED,
    shuffle      = True,
)
print(f"✅  Split — train: {len(train_pairs):,}  val: {len(val_pairs):,}")
print(f"    Train kin ratio: {sum(l for _,_,l in train_pairs) / len(train_pairs):.3f}")
print(f"    Val   kin ratio: {sum(l for _,_,l in val_pairs)   / len(val_pairs):.3f}")


class KinshipPairDataset(Dataset):
    """
    Each sample: (emb_a, emb_b, label)  —  embeddings loaded from cache.
    Training augmentation: pair-swap symmetry + light L2-normalized noise.
    """

    def __init__(self, pairs: list, cache: dict, augment: bool = False):
        self.pairs   = pairs
        self.cache   = cache
        self.augment = augment

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_a, img_b, label = self.pairs[idx]
        # NEW — handles both numpy arrays and Tensors
        def _to_tensor(x):
            if isinstance(x, torch.Tensor):
                return x.clone().float()
            return torch.from_numpy(np.array(x, dtype=np.float32))

        emb_a = _to_tensor(self.cache[img_a])
        emb_b = _to_tensor(self.cache[img_b])
        if self.augment:
            if torch.rand(1).item() < 0.5:
                emb_a, emb_b = emb_b, emb_a           # symmetry augmentation
            if torch.rand(1).item() < 0.4:
                emb_a = F.normalize(emb_a + torch.randn_like(emb_a) * 0.020, p=2, dim=0)
                emb_b = F.normalize(emb_b + torch.randn_like(emb_b) * 0.020, p=2, dim=0)

        return emb_a, emb_b, torch.tensor([float(label)], dtype=torch.float32)


# Weighted sampler: guarantees equal kin/non-kin representation per batch
_train_labels = [l for _, _, l in train_pairs]
_class_counts = [_train_labels.count(c) for c in (0, 1)]
_weights      = [1.0 / _class_counts[l] for l in _train_labels]
_sampler      = WeightedRandomSampler(_weights, num_samples=len(_train_labels), replacement=True)

train_ds = KinshipPairDataset(train_pairs, EMBED_CACHE, augment=True)
val_ds   = KinshipPairDataset(val_pairs,   EMBED_CACHE, augment=False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=_sampler,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True)

print(f"✅  DataLoaders — {len(train_loader)} train batches / {len(val_loader)} val batches")

✅  Split — train: 1,600  val: 400
    Train kin ratio: 0.200
    Val   kin ratio: 0.200
✅  DataLoaders — 50 train batches / 7 val batches


In [10]:
# ──────────────────────────────────────────────────────────────────────────
# Siamese KinshipHead
# ──────────────────────────────────────────────────────────────────────────

class KinshipHead(nn.Module):
    """
    4-component Siamese fusion head.

    Fusion  : [|a−b|, a⊙b, cos(a,b), ‖a−b‖₂]  → shape [B, D*2+2]
    MLP     : Linear → BN → GELU → Dropout  (repeated per hidden layer)
    Output  : single logit (BCEWithLogitsLoss compatible)
    """

    def __init__(self, embed_dim: int, hidden_dims: list, dropout: float):
        super().__init__()
        in_dim = embed_dim * 2 + 2   # 512+512+1+1 = 1026

        layers: list[nn.Module] = []
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h, bias=False),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(p=dropout),
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.head = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def _fuse(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        diff    = torch.abs(a - b)
        product = a * b
        cos     = F.cosine_similarity(a, b).unsqueeze(1)
        l2      = torch.norm(diff, dim=1, keepdim=True)
        return torch.cat([diff, product, cos, l2], dim=1)

    def forward(self, emb_a: torch.Tensor, emb_b: torch.Tensor) -> torch.Tensor:
        return self.head(self._fuse(emb_a, emb_b))


MODEL = KinshipHead(
    embed_dim   = CFG.EMBED_DIM,
    hidden_dims = CFG.HEAD_HIDDEN,
    dropout     = CFG.HEAD_DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in MODEL.parameters())
print(f"✅  KinshipHead — {n_params:,} trainable parameters")
print(MODEL)

✅  KinshipHead — 65,857 trainable parameters
KinshipHead(
  (head): Sequential(
    (0): Linear(in_features=1026, out_features=64, bias=False)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.6, inplace=False)
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [11]:
# ──────────────────────────────────────────────────────────────────────────
# Loss, Optimizer & Scheduler
# ──────────────────────────────────────────────────────────────────────────

class LabelSmoothBCE(nn.Module):
    """BCE with label smoothing to prevent over-confident predictions."""

    def __init__(self, smoothing: float = 0.05):
        super().__init__()
        self.s   = smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        smooth_targets = targets * (1 - self.s) + 0.5 * self.s
        return self.bce(logits, smooth_targets)


criterion = LabelSmoothBCE(smoothing=0.05)
optimizer = torch.optim.AdamW(
    MODEL.parameters(),
    lr           = CFG.LR,
    weight_decay = CFG.WEIGHT_DECAY,
    betas        = (0.9, 0.999),
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode     = "max",       # maximize AUC
    factor   = CFG.LR_FACTOR,
    patience = CFG.LR_PATIENCE,
    min_lr   = 1e-6,
)

print(f"✅  Loss      : LabelSmoothBCE(s=0.05)")
print(f"✅  Optimizer : AdamW(lr={CFG.LR}, wd={CFG.WEIGHT_DECAY})")
print(f"✅  Scheduler : ReduceLROnPlateau(mode=max, patience={CFG.LR_PATIENCE})")

✅  Loss      : LabelSmoothBCE(s=0.05)
✅  Optimizer : AdamW(lr=0.001, wd=0.01)
✅  Scheduler : ReduceLROnPlateau(mode=max, patience=10)


In [12]:
# ──────────────────────────────────────────────────────────────────────────
# Training Loop
# ──────────────────────────────────────────────────────────────────────────

from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score, f1_score,
    roc_curve, classification_report, confusion_matrix,
)
import time


def run_epoch(
    model     : nn.Module,
    loader    : DataLoader,
    criterion : nn.Module,
    optimizer : torch.optim.Optimizer | None,
    device    : torch.device,
    is_train  : bool,
) -> dict:
    model.train(is_train)
    ctx = torch.enable_grad if is_train else torch.no_grad

    total_loss = 0.0
    all_labels : list = []
    all_probs  : list = []

    with ctx():
        for emb_a, emb_b, labels in loader:
            emb_a  = emb_a.to(device, non_blocking=True)
            emb_b  = emb_b.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(emb_a, emb_b)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * emb_a.size(0)
            all_labels.extend(labels.squeeze(1).cpu().numpy().tolist())
            all_probs.extend(
                torch.sigmoid(logits).squeeze(1).detach().cpu().numpy().tolist()
            )

    labels_arr = np.array(all_labels, dtype=int)
    probs_arr  = np.array(all_probs,  dtype=float)
    avg_loss   = total_loss / len(labels_arr)

    # AUC
    auc = roc_auc_score(labels_arr, probs_arr) \
          if len(np.unique(labels_arr)) > 1 else 0.5

    # Youden's J threshold → maximizes (TPR − FPR) = sensitivity + specificity − 1
    fpr, tpr, thresholds = roc_curve(labels_arr, probs_arr)
    best_idx  = int(np.argmax(tpr - fpr))
    threshold = float(thresholds[best_idx])

    preds  = (probs_arr >= threshold).astype(int)
    acc    = float(accuracy_score(labels_arr, preds))
    recall = float(recall_score(labels_arr, preds, zero_division=0))
    f1     = float(f1_score(labels_arr,      preds, zero_division=0))

    return {
        "loss"     : avg_loss,
        "auc"      : auc,
        "acc"      : acc,
        "recall"   : recall,
        "f1"       : f1,
        "threshold": threshold,
        "_probs"   : probs_arr,
        "_labels"  : labels_arr,
    }


def train(
    model      : nn.Module,
    train_ldr  : DataLoader,
    val_ldr    : DataLoader,
    criterion  : nn.Module,
    optimizer  : torch.optim.Optimizer,
    scheduler,
    cfg        : Config,
    device     : torch.device,
) -> list:
    best_auc   = 0.0
    no_improve = 0
    history    = []

    print("=" * 80)
    print("  TRAINING START")
    print("=" * 80)

    for epoch in range(1, cfg.NUM_EPOCHS + 1):
        t0     = time.time()
        tr     = run_epoch(model, train_ldr, criterion, optimizer, device, True)
        vl     = run_epoch(model, val_ldr,   criterion, None,      device, False)
        elapsed = time.time() - t0

        scheduler.step(vl["auc"])
        history.append({"epoch": epoch, "train": tr, "val": vl})

        improved = vl["auc"] > best_auc
        marker   = " ✓" if improved else ""

        print(
            f"Ep {epoch:>4}/{cfg.NUM_EPOCHS}  "
            f"T  loss={tr['loss']:.4f}  AUC={tr['auc']:.4f}  Rec={tr['recall']:.4f}  "
            f"|  V  loss={vl['loss']:.4f}  AUC={vl['auc']:.4f}  "
            f"Acc={vl['acc']:.4f}  Rec={vl['recall']:.4f}  "
            f"({elapsed:.1f}s){marker}"
        )

        if improved:
            best_auc   = vl["auc"]
            no_improve = 0
            torch.save({
                "epoch"      : epoch,
                "model_state": model.state_dict(),
                "opt_state"  : optimizer.state_dict(),
                "best_auc"   : best_auc,
                "threshold"  : vl["threshold"],
            }, cfg.BEST_MODEL)
        else:
            no_improve += 1
            if no_improve >= cfg.ES_PATIENCE:
                print(f"\n⏹   Early stopping — no improvement for {cfg.ES_PATIENCE} epochs.")
                break

    print(f"\n✅  Training complete — Best Val AUC: {best_auc:.4f}")
    return history


HISTORY = train(
    MODEL, train_loader, val_loader,
    criterion, optimizer, scheduler,
    CFG, DEVICE,
)

  TRAINING START
Ep    1/200  T  loss=0.7233  AUC=0.6059  Rec=0.6770  |  V  loss=0.6990  AUC=0.5427  Acc=0.6975  Rec=0.3000  (1.0s) ✓
Ep    2/200  T  loss=0.5954  AUC=0.7605  Rec=0.6765  |  V  loss=0.5887  AUC=0.5692  Acc=0.4900  Rec=0.6875  (0.7s) ✓
Ep    3/200  T  loss=0.4946  AUC=0.8633  Rec=0.7439  |  V  loss=0.6391  AUC=0.5879  Acc=0.5625  Rec=0.6125  (0.8s) ✓
Ep    4/200  T  loss=0.4443  AUC=0.9020  Rec=0.8303  |  V  loss=0.6019  AUC=0.6059  Acc=0.5075  Rec=0.7750  (0.8s) ✓
Ep    5/200  T  loss=0.3893  AUC=0.9352  Rec=0.8994  |  V  loss=0.6129  AUC=0.6075  Acc=0.6150  Rec=0.5875  (0.8s) ✓
Ep    6/200  T  loss=0.3865  AUC=0.9328  Rec=0.8753  |  V  loss=0.6483  AUC=0.5908  Acc=0.5725  Rec=0.6125  (0.6s)
Ep    7/200  T  loss=0.3476  AUC=0.9523  Rec=0.8606  |  V  loss=0.6382  AUC=0.6031  Acc=0.6550  Rec=0.5625  (0.6s)
Ep    8/200  T  loss=0.3255  AUC=0.9613  Rec=0.9089  |  V  loss=0.6408  AUC=0.6108  Acc=0.6400  Rec=0.5625  (0.6s) ✓
Ep    9/200  T  loss=0.3054  AUC=0.9697  Rec=0.9017

In [13]:
# ──────────────────────────────────────────────────────────────────────────
# Final Evaluation
# ──────────────────────────────────────────────────────────────────────────

# Load best checkpoint
ckpt = torch.load(CFG.BEST_MODEL, map_location=DEVICE, weights_only=False)
MODEL.load_state_dict(ckpt["model_state"])
threshold = ckpt["threshold"]
print(f"✅  Best model restored — epoch={ckpt['epoch']}  AUC={ckpt['best_auc']:.4f}  "
      f"threshold={threshold:.4f} (Youden's J)")

# Full val-set evaluation
final = run_epoch(MODEL, val_loader, criterion, None, DEVICE, is_train=False)
preds = (final["_probs"] >= threshold).astype(int)

print(f"\n{'═'*60}")
print("  FINAL VALIDATION RESULTS")
print(f"{'═'*60}")
print(f"  AUC-ROC   : {final['auc']:.4f}")
print(f"  Threshold : {threshold:.4f}  (Youden's J on validation)")
print(f"{'─'*60}")
print(classification_report(
    final["_labels"], preds,
    target_names=["Non-kin", "Kin"],
    digits=4,
))
print("Confusion Matrix:")
print(confusion_matrix(final["_labels"], preds))
print(f"{'═'*60}")

✅  Best model restored — epoch=34  AUC=0.6761  threshold=0.4845 (Youden's J)

════════════════════════════════════════════════════════════
  FINAL VALIDATION RESULTS
════════════════════════════════════════════════════════════
  AUC-ROC   : 0.6761
  Threshold : 0.4845  (Youden's J on validation)
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

     Non-kin     0.8724    0.7906    0.8295       320
         Kin     0.3909    0.5375    0.4526        80

    accuracy                         0.7400       400
   macro avg     0.6317    0.6641    0.6411       400
weighted avg     0.7761    0.7400    0.7541       400

Confusion Matrix:
[[253  67]
 [ 37  43]]
════════════════════════════════════════════════════════════


In [14]:
# ── Download FIW dataset (thesistime/fiwdata) ─────────────────────────────
import os
from google.colab import files

# Step 1: Upload kaggle.json
files.upload()  # select kaggle.json

# Step 2: Setup credentials
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Step 3: Download — dataset slug is thesistime/fiwdata
!kaggle datasets download -d thesistime/fiwdata -p /content/fiw

# Step 4: Unzip
!unzip -q /content/fiw/fiwdata.zip -d /content/fiw
print("Done!", os.listdir("/content/fiw"))

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/thesistime/fiwdata
License(s): unknown
100% 388M/388M [00:01<00:00, 208MB/s]

Done! ['train-faces', 'test-private-labels', 'sample_submission.csv', 'test-public-faces', 'train_relationships.csv', 'fiwdata.zip', 'test', 'test-public-lists', 'test-private-lists', 'README.md', 'test-private-faces', 'train', 'test-faces']


In [15]:
import glob, random
import pandas as pd

def load_fiw_pairs(fiw_dir: str, cache: dict, neg_ratio: float = 1.0) -> list:
    csv_path   = os.path.join(fiw_dir, "train_relationships.csv")
    train_root = os.path.join(fiw_dir, "train")

    df = pd.read_csv(csv_path)   # columns: p1, p2 — all are kin pairs

    # Build member-key → image paths
    mid_to_imgs = {}
    for fam_dir in sorted(glob.glob(os.path.join(train_root, "F*"))):
        fam = os.path.basename(fam_dir)
        for mid_dir in sorted(glob.glob(os.path.join(fam_dir, "MID*"))):
            mid  = os.path.basename(mid_dir)
            key  = f"{fam}/{mid}"
            imgs = sorted(glob.glob(os.path.join(mid_dir, "*.jpg")) +
                          glob.glob(os.path.join(mid_dir, "*.png")))
            if imgs:
                mid_to_imgs[key] = imgs

    print(f"Members with images: {len(mid_to_imgs)}")

    # ── Positive pairs (all rows in CSV are kin) ──────────────────────────
    pos_pairs = []
    for _, row in df.iterrows():
        p1 = str(row["p1"]).strip()
        p2 = str(row["p2"]).strip()
        for ia in mid_to_imgs.get(p1, []):
            for ib in mid_to_imgs.get(p2, []):
                if ia in cache and ib in cache:
                    pos_pairs.append((ia, ib, 1))

    # ── Negative pairs (random cross-family sampling) ─────────────────────
    all_keys  = list(mid_to_imgs.keys())
    n_neg     = int(len(pos_pairs) * neg_ratio)
    neg_pairs = []
    attempts  = 0
    while len(neg_pairs) < n_neg and attempts < n_neg * 15:
        k1, k2 = random.sample(all_keys, 2)
        if k1.split("/")[0] == k2.split("/")[0]:  # same family → skip
            attempts += 1
            continue
        ia = random.choice(mid_to_imgs[k1])
        ib = random.choice(mid_to_imgs[k2])
        if ia in cache and ib in cache:
            neg_pairs.append((ia, ib, 0))
        attempts += 1

    all_pairs = pos_pairs + neg_pairs
    random.shuffle(all_pairs)
    print(f"✅  FIW → {len(pos_pairs)} kin + {len(neg_pairs)} non-kin = {len(all_pairs)} total pairs")
    return all_pairs

In [17]:
# ── Redefine extract function ──────────────────────────────────────────────
from PIL import Image
import torchvision.transforms as T
from tqdm import tqdm

_transform = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def extract(backbone: nn.Module, img_paths: list, device: str,
            batch_size: int = CFG.EXTRACT_BATCH) -> dict:
    """Batch-extract L2-normalised embeddings → {path: tensor} dict."""
    backbone.eval()
    cache = {}
    for i in tqdm(range(0, len(img_paths), batch_size), desc="Extracting"):
        batch_paths = img_paths[i : i + batch_size]
        imgs = []
        valid_paths = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                imgs.append(_transform(img))
                valid_paths.append(p)
            except Exception:
                pass   # skip corrupt images silently
        if not imgs:
            continue
        tensor = torch.stack(imgs).to(device)
        with torch.no_grad():
            out  = backbone(tensor)
            emb  = out[0] if isinstance(out, (list, tuple)) else out
            if isinstance(emb, dict):
                emb = next(iter(emb.values()))
            emb  = F.normalize(emb, p=2, dim=1).cpu()
        for path, vec in zip(valid_paths, emb):
            cache[path] = vec
    return cache

print("✅  extract() function ready.")

✅  extract() function ready.


In [18]:
FIW_CACHE_PATH = "/content/cache_fiw_embeddings.npy"
FIW_DIR        = "/content/fiw"

all_fiw_imgs = sorted(
    glob.glob(f"{FIW_DIR}/train/**/*.jpg", recursive=True) +
    glob.glob(f"{FIW_DIR}/train/**/*.png", recursive=True)
)
print(f"Total FIW images found: {len(all_fiw_imgs)}")

if os.path.exists(FIW_CACHE_PATH):
    FIW_CACHE = np.load(FIW_CACHE_PATH, allow_pickle=True).item()
    missing   = [p for p in all_fiw_imgs if p not in FIW_CACHE]
    print(f"Cache exists: {len(FIW_CACHE)} embeddings, {len(missing)} missing")
    if missing:
        new_embs = extract(BACKBONE, missing, DEVICE)
        FIW_CACHE.update(new_embs)
        np.save(FIW_CACHE_PATH, FIW_CACHE)
else:
    print("Extracting from scratch — this takes ~30–40 min on T4...")
    FIW_CACHE = extract(BACKBONE, all_fiw_imgs, DEVICE)
    np.save(FIW_CACHE_PATH, FIW_CACHE)

print(f"✅  FIW cache ready: {len(FIW_CACHE)} embeddings")

Total FIW images found: 12379
Extracting from scratch — this takes ~30–40 min on T4...


Extracting: 100%|██████████| 97/97 [01:20<00:00,  1.20it/s]


✅  FIW cache ready: 12379 embeddings


In [19]:
from sklearn.model_selection import train_test_split

# ── Stage 1: Pretrain on FIW ──────────────────────────────────────────────
FIW_PAIRS  = load_fiw_pairs(FIW_DIR, FIW_CACHE, neg_ratio=1.0)
fiw_labels = [l for _, _, l in FIW_PAIRS]

fiw_tr, fiw_vl = train_test_split(FIW_PAIRS, test_size=0.1,
                                   stratify=fiw_labels, random_state=CFG.SEED)

fiw_train_loader = DataLoader(KinshipPairDataset(fiw_tr, FIW_CACHE, augment=True),
                               batch_size=CFG.BATCH_SIZE, shuffle=True,
                               num_workers=CFG.NUM_WORKERS, pin_memory=True)
fiw_val_loader   = DataLoader(KinshipPairDataset(fiw_vl, FIW_CACHE, augment=False),
                               batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                               num_workers=CFG.NUM_WORKERS, pin_memory=True)

# Fresh model
MODEL     = KinshipHead(CFG.EMBED_DIM, CFG.HEAD_HIDDEN, CFG.HEAD_DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(MODEL.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
              factor=CFG.LR_FACTOR, patience=CFG.LR_PATIENCE, min_lr=1e-6)
criterion = LabelSmoothBCE(smoothing=0.05)

print("=" * 60)
print("  STAGE 1 — Pretraining on FIW")
print("=" * 60)
train(MODEL, fiw_train_loader, fiw_val_loader, criterion, optimizer, scheduler, CFG, DEVICE)
torch.save(MODEL.state_dict(), "/content/checkpoints/head_fiw_pretrained.pt")
print("✅  FIW pretrained head saved.")


# ── Stage 2: Fine-tune on KinFaceW-II ─────────────────────────────────────
MODEL.load_state_dict(torch.load("/content/checkpoints/head_fiw_pretrained.pt"))

optimizer = torch.optim.AdamW(MODEL.parameters(),
                               lr=CFG.LR * 0.3,          # lower LR for fine-tune
                               weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
              factor=CFG.LR_FACTOR, patience=CFG.LR_PATIENCE, min_lr=1e-6)

print("=" * 60)
print("  STAGE 2 — Fine-tuning on KinFaceW-II")
print("=" * 60)
train(MODEL, train_loader, val_loader, criterion, optimizer, scheduler, CFG, DEVICE)

Members with images: 2316
✅  FIW → 165179 kin + 165179 non-kin = 330358 total pairs
  STAGE 1 — Pretraining on FIW
  TRAINING START
Ep    1/200  T  loss=0.5500  AUC=0.8057  Rec=0.6713  |  V  loss=0.4942  AUC=0.8569  Acc=0.7780  Rec=0.7155  (67.0s) ✓
Ep    2/200  T  loss=0.5234  AUC=0.8311  Rec=0.6843  |  V  loss=0.4756  AUC=0.8722  Acc=0.7917  Rec=0.7355  (73.4s) ✓
Ep    3/200  T  loss=0.5133  AUC=0.8403  Rec=0.6924  |  V  loss=0.4630  AUC=0.8811  Acc=0.8022  Rec=0.7424  (68.0s) ✓
Ep    4/200  T  loss=0.5077  AUC=0.8451  Rec=0.7021  |  V  loss=0.4571  AUC=0.8864  Acc=0.8057  Rec=0.7613  (68.1s) ✓
Ep    5/200  T  loss=0.5020  AUC=0.8504  Rec=0.7179  |  V  loss=0.4547  AUC=0.8899  Acc=0.8082  Rec=0.7891  (68.4s) ✓
Ep    6/200  T  loss=0.5005  AUC=0.8512  Rec=0.7185  |  V  loss=0.4496  AUC=0.8931  Acc=0.8108  Rec=0.7742  (71.3s) ✓
Ep    7/200  T  loss=0.4979  AUC=0.8538  Rec=0.7110  |  V  loss=0.4471  AUC=0.8959  Acc=0.8151  Rec=0.7666  (69.6s) ✓
Ep    8/200  T  loss=0.4954  AUC=0.8557  R

KeyboardInterrupt: 

In [24]:
from google.colab import files
files.download('/content/checkpoints/best_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>